# Module 8 — Grad Café: Data Preparation & Exploratory Statistics

This notebook extends the Module 7 cloud pipeline. It **loads the Grad Café JSON
dataset from Amazon S3** into this SageMaker instance (via the Module 7 `boto3`
workflow in `s3_fetch.py`), then **cleans, validates, feature-engineers, and
statistically analyzes** it with **Pandas, NumPy, SciPy, and Matplotlib**, and
finally **writes the cleaned dataset back to S3**.

It runs **top-to-bottom in order**. Reusable, repetitive helpers (rendering
tables to PNG, the six plots, the analytics PDF) live in `gradcafe.py`; all of
the substantive data work is shown inline below. No final answers are hard-coded
— every reported number is computed from the data.

**Setup (run once):** `pip install -r requirements.txt`

In [1]:
# Core scientific stack + our Module 7/8 helpers.
import json

import numpy as np
import pandas as pd
from scipy import stats

import s3_fetch      # Module 7 boto3 download + new upload helper
import gradcafe      # PNG-table renderer, plots, analytics-PDF builder

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("pandas", pd.__version__, "| numpy", np.__version__)

Matplotlib is building the font cache; this may take a moment.


pandas 2.3.3 | numpy 2.0.2


## 1. Load the Grad Café data from S3 (boto3)

`s3_fetch.download_dataset()` builds an S3 client from **boto3's default
credential chain** — on SageMaker that is the notebook instance's attached **IAM
execution role**, so no keys are hard-coded. It downloads `applicant_data.json`
from the private bucket and saves it locally as `applicant_data_SM.json`.

In [2]:
local_path = s3_fetch.download_dataset()      # S3 -> applicant_data_SM.json
df = pd.read_json(local_path)                 # raw JSON -> DataFrame
raw_df = df.copy()                            # preserved raw copy (before any cleaning)
print("Loaded", local_path, "->", raw_df.shape[0], "rows x", raw_df.shape[1], "cols")

Loaded applicant_data_SM.json -> 30000 rows x 18 cols


### First 5 rows of the raw dataset
Saved as `initial_dataframe.png`.

In [3]:
gradcafe.render_table_png(raw_df, "initial_dataframe.png",
                          title="Raw Grad Café dataset — first 5 rows")
raw_df.head()

,university,program,raw_program,degree,status,notification_date,date_added,semester,year,student_type,url,comments,gpa,gre,gre_v,gre_aw,llm-generated-program,llm-generated-university
0,University of Virginia,Speech Language Pathology,Speech Language Pathology Masters,Masters,Accepted,Feb 10,"Jun 02, 2026",Fall,2026,American,https://www.thegradcafe.com/result/1020294,"Out-of-field major, 4.0 in CSD pre-reqs. 4 str...",3.54,NaN,NaN,NaN,Speech-Language Pathology,University of Virginia
1,Appalachian State University,Speech Language Pathology,Speech Language Pathology Masters,Masters,Accepted,Jun 01,"Jun 01, 2026",Fall,2026,American,https://www.thegradcafe.com/result/1020293,,3.44,NaN,NaN,NaN,Speech-Language Pathology,Appalachian State University
2,Case Western Reserve University,Chemistry,Chemistry PhD,PhD,Rejected,May 19,"Jun 01, 2026",Fall,2026,International,https://www.thegradcafe.com/result/1020292,,NaN,NaN,NaN,NaN,Chemistry,Case Western Reserve University
3,Georgia State University,Chemistry,Chemistry PhD,PhD,Rejected,May 29,"Jun 01, 2026",Fall,2026,International,https://www.thegradcafe.com/result/1020291,Submitted my application in November before th...,NaN,NaN,NaN,NaN,Chemistry,Georgia State University
4,Northeastern University,Mathematics,Mathematics PhD,PhD,Rejected,May 18,"Jun 01, 2026",Fall,2026,American,https://www.thegradcafe.com/result/1020290,Very difficult to get in touch with anyone at ...,3.68,NaN,NaN,NaN,Mathematics,Northeastern University


## 2. Raw inspection & missingness

Report the raw shape, column names, and dtype of every column, then build a
missingness table (count + percentage) using NumPy for the percentage math.

In [4]:
print("Rows:", raw_df.shape[0])
print("Columns:", raw_df.shape[1])
print("Column names:", list(raw_df.columns))
print()
print("Datatypes:")
print(raw_df.dtypes)

Rows: 30000
Columns: 18
Column names: ['university', 'program', 'raw_program', 'degree', 'status', 'notification_date', 'date_added', 'semester', 'year', 'student_type', 'url', 'comments', 'gpa', 'gre', 'gre_v', 'gre_aw', 'llm-generated-program', 'llm-generated-university']

Datatypes:
university                   object
program                      object
raw_program                  object
degree                       object
status                       object
notification_date            object
date_added                   object
semester                     object
year                          int64
student_type                 object
url                          object
comments                     object
gpa                         float64
gre                         float64
gre_v                       float64
gre_aw                      float64
llm-generated-program        object
llm-generated-university     object
dtype: object


In [5]:
# Missingness for EVERY column: count + NumPy-computed percentage.
missing_count = raw_df.isna().sum()
missing_pct = np.round(missing_count.to_numpy() / len(raw_df) * 100, 2)
missingness = pd.DataFrame({
    "missing_count": missing_count.to_numpy(),
    "missing_pct": missing_pct,
}, index=raw_df.columns).sort_values("missing_count", ascending=False)

missingness.to_csv("missingness_summary.csv")
gradcafe.render_table_png(missingness, "missingness_summary.png",
                          title="Missingness summary (per column)",
                          max_rows=len(missingness))
missingness

,missing_count,missing_pct
gre_aw,28114,93.71
gre_v,27993,93.31
gre,27602,92.01
gpa,11906,39.69
university,0,0.00
program,0,0.00
llm-generated-program,0,0.00
comments,0,0.00
url,0,0.00
student_type,0,0.00


## 3. Clean & standardize

We now build the analysis columns using the assignment's canonical names
(`Program`, `University`, `Degree`, `US/International`, `GPA`, `GRE`, `GRE V`,
`GRE AW`, `term`, `outcome`, `decision_date`). Drop counts are tracked per rule
so we can report them in the validation step.

We start from the raw copy and **remove rows where `program` is None**.

In [6]:
n_raw = len(raw_df)
work = raw_df.copy()

# Rule 1 — drop rows with no program.
before = len(work)
work = work[work["program"].notna()].copy()
dropped_program_none = before - len(work)
print(f"Dropped {dropped_program_none} rows where program is None")

Dropped 0 rows where program is None


### Split the combined program field into `Program` and `University`

Some Grad Café exports pack the university into the program string
(`"Computer Science, Stanford University"`); ours already ships a separate
`university` column. The code handles **both**: it splits on a comma when present
and otherwise uses the explicit `university` field. Text is standardized —
leading/trailing whitespace stripped and repeated internal whitespace collapsed —
while meaningful capitalization is preserved.

In [7]:
def clean_text(series):
    """Strip ends + collapse internal whitespace; preserve capitalization."""
    return (series.astype("string")
                  .str.strip()
                  .str.replace(r"\s+", " ", regex=True))

parts = work["program"].astype("string").str.split(r"\s*,\s*", n=1, expand=True)
work["Program"] = clean_text(parts[0])
uni_from_split = parts[1] if parts.shape[1] > 1 else pd.Series(pd.NA, index=work.index)
work["University"] = clean_text(work["university"].astype("string").fillna(uni_from_split))
work[["Program", "University"]].head()

,Program,University
0,Speech Language Pathology,University of Virginia
1,Speech Language Pathology,Appalachian State University
2,Chemistry,Case Western Reserve University
3,Chemistry,Georgia State University
4,Mathematics,Northeastern University


### Low-frequency programs

Count occurrences of each `Program`, then show only the programs that occur
**fewer than 3 times**. Saved as `low_program_count.png`.

In [8]:
program_counts = work["Program"].value_counts()
low_program = program_counts[program_counts < 3]
print(f"{len(low_program)} programs occur fewer than 3 times "
      f"(out of {program_counts.size} unique programs)")
gradcafe.render_table_png(low_program.rename("count"), "low_program_count.png",
                          title="Programs occurring fewer than 3 times (head)",
                          max_rows=20)
low_program.head(20)

2202 programs occur fewer than 3 times (out of 2910 unique programs)


Program
Applied Statistics and Research Methods    2
Energy Science and Technology              2
Systems Design Engineering                 2
Physics and Astronomy                      2
School-Clinical Psychology                 2
che                                        2
Winterthur Program                         2
social work                                2
Art and Museum Studies                     2
Rhetoric and Professional Communication    2
Modern Culture and Media                   2
Computational Mathematics                  2
Mechanical and Mechatronics Engineering    2
Curatorial Studies                         2
Museums                                    2
Interdisciplinary Life Sciences (PULSe)    2
Medical Sciences                           2
Applied Physiology and Kinesiology         2
Physiology                                 2
Historical Archaeology                     2
Name: count, dtype: Int64

### Parse dates and split `Status` into `outcome` + `decision_date`

`date_added` becomes a real datetime. For the decision, we build a `term` from
`semester`+`year`, extract the **outcome** keyword from the status field
(mapping `Interview → Interviewed`), and read the decision **date**. We handle
both an inline `"Accepted on 15 Feb"` style *and* a separate `notification_date`
column, then **infer the decision year from `term`** (e.g. `Fall 2026 → 2026`).

In [9]:
# date_added -> datetime
work["date_added"] = pd.to_datetime(work["date_added"], format="%b %d, %Y",
                                    errors="coerce")

# term from semester + year
work["term"] = (work["semester"].astype("string").str.strip() + " " +
                work["year"].astype("string").str.strip()).str.strip()

# outcome: restricted to the four allowed labels
outcome_map = {"Accept": "Accepted", "Reject": "Rejected",
               "Wait": "Waitlisted", "Interview": "Interviewed"}
status = work["status"].fillna("").astype(str)
work["outcome"] = (status.str.extract(r"(Accept|Reject|Wait|Interview)", expand=False)
                         .map(outcome_map))

# decision date string: prefer an inline "... on <date>", else notification_date
inline = status.str.extract(r"on\s+(.+)$", expand=False)
date_str = inline.fillna(work["notification_date"]).astype("string").str.strip()

# infer the year from term, then parse "<Mon DD>, <YYYY>"
year = work["term"].astype("string").str.extract(r"(\d{4})", expand=False)
work["decision_date"] = pd.to_datetime(date_str + ", " + year,
                                       format="%b %d, %Y", errors="coerce")

work[["term", "status", "outcome", "notification_date", "decision_date",
      "date_added"]].head()

,term,status,outcome,notification_date,decision_date,date_added
0,Fall 2026,Accepted,Accepted,Feb 10,2026-02-10,2026-06-02
1,Fall 2026,Accepted,Accepted,Jun 01,2026-06-01,2026-06-01
2,Fall 2026,Rejected,Rejected,May 19,2026-05-19,2026-06-01
3,Fall 2026,Rejected,Rejected,May 29,2026-05-29,2026-06-01
4,Fall 2026,Rejected,Rejected,May 18,2026-05-18,2026-06-01


Saved as `date_based.png`.

In [10]:
gradcafe.render_table_png(
    work[["Program", "term", "status", "outcome", "decision_date", "date_added"]],
    "date_based.png", title="After date parsing (outcome + decision_date)")
work[["outcome", "decision_date"]].head()

,outcome,decision_date
0,Accepted,2026-02-10
1,Accepted,2026-06-01
2,Rejected,2026-05-19
3,Rejected,2026-05-29
4,Rejected,2026-05-18


### Standardize `Degree` and `US/International`

`Degree` is restricted to **Master's / PhD / PsyD** (rows with any other value —
MFA, JD, EdD, MBA, Other — are dropped). `US/International` is restricted to
**International / American / Other**; blanks and unexpected values are mapped to
`Other`.

In [11]:
degree_map = {"Masters": "Master's", "Master's": "Master's",
              "PhD": "PhD", "PsyD": "PsyD"}
work["Degree"] = work["degree"].map(degree_map)

# Rule 2 — drop invalid degrees (those not in the map become NaN).
before = len(work)
invalid_degree = int(work["Degree"].isna().sum())
work = work[work["Degree"].notna()].copy()
dropped_bad_degree = before - len(work)
print(f"Dropped {dropped_bad_degree} rows with an invalid Degree value")

allowed_intl = {"International", "American", "Other"}
st = work["student_type"].astype("string").str.strip().replace("", "Other")
work["US/International"] = st.where(st.isin(allowed_intl), "Other")
print(work["US/International"].value_counts(dropna=False).to_dict())

Dropped 859 rows with an invalid Degree value
{'American': 14943, 'International': 13565, 'Other': 633}


### Convert score columns to floats

`GPA`, `GRE`, `GRE V`, `GRE AW` are coerced to numeric floats; unparseable
values become `NaN`. Saved as `float_columns.png`.

In [12]:
num_src = {"GPA": "gpa", "GRE": "gre", "GRE V": "gre_v", "GRE AW": "gre_aw"}
for new, old in num_src.items():
    work[new] = pd.to_numeric(work[old], errors="coerce")

print(work[["GPA", "GRE", "GRE V", "GRE AW"]].dtypes)
gradcafe.render_table_png(work[["GPA", "GRE", "GRE V", "GRE AW"]],
                          "float_columns.png",
                          title="Numeric-converted columns (float64)")
work[["GPA", "GRE", "GRE V", "GRE AW"]].head()

GPA       float64
GRE       float64
GRE V     float64
GRE AW    float64
dtype: object


,GPA,GRE,GRE V,GRE AW
0,3.54,NaN,NaN,NaN
1,3.44,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,3.68,NaN,NaN,NaN


### Data-validation report

Rows removed during cleaning (overall and by rule), the final cleaned shape, and
a raw-vs-cleaned comparison.

In [13]:
cleaned = work
validation = pd.DataFrame({
    "metric": ["raw rows", "dropped: program is None", "dropped: invalid Degree",
               "cleaned rows", "total rows removed", "columns (raw)",
               "columns (cleaned)"],
    "value": [n_raw, dropped_program_none, dropped_bad_degree, len(cleaned),
              n_raw - len(cleaned), raw_df.shape[1], cleaned.shape[1]],
})
print(validation.to_string(index=False))
print()
print("Raw vs cleaned row count:", n_raw, "->", len(cleaned),
      f"({len(cleaned)/n_raw*100:.1f}% retained)")

                  metric  value
                raw rows  30000
dropped: program is None      0
 dropped: invalid Degree    859
            cleaned rows  29141
      total rows removed    859
           columns (raw)     18
       columns (cleaned)     29

Raw vs cleaned row count: 30000 -> 29141 (97.1% retained)


## 4. Engineered analytical columns

All created with vectorized Pandas/NumPy (`np.where`, `np.select`) — no loops.

* **has_valid_gpa** — 1 if GPA present and in [0, 4.0], else 0.
* **has_valid_gre / has_valid_gre_v** — 1 if the score is present and > 0.
* **days_to_decision** — `decision_date − date_added` in days (NaN if no decision
  date). *Note:* because the decision year is inferred from `term`, this can be
  negative when the notification month precedes the added month — a known
  limitation discussed in `analytics.pdf`.
* **decision_speed** — `np.select` buckets: 0-30 / 31-60 / 61+ days, else Unknown
  (covers missing and negative gaps).
* **application_season** — from the `date_added` month: Aug-Nov = Early Cycle,
  Dec-Feb = Mid Cycle, Mar-Jul = Late Cycle.

In [14]:
cleaned["has_valid_gpa"] = np.where(cleaned["GPA"].between(0, 4.0), 1, 0)
cleaned["has_valid_gre"] = np.where(cleaned["GRE"] > 0, 1, 0)
cleaned["has_valid_gre_v"] = np.where(cleaned["GRE V"] > 0, 1, 0)

cleaned["days_to_decision"] = (cleaned["decision_date"] - cleaned["date_added"]).dt.days

d = cleaned["days_to_decision"]
speed_conds = [d.between(0, 30), d.between(31, 60), d > 60]
cleaned["decision_speed"] = np.select(
    speed_conds, ["0-30 days", "31-60 days", "61+ days"], default="Unknown")

month = cleaned["date_added"].dt.month
season_conds = [month.between(8, 11), month.isin([12, 1, 2]), month.between(3, 7)]
cleaned["application_season"] = np.select(
    season_conds, ["Early Cycle", "Mid Cycle", "Late Cycle"], default="Unknown")

cleaned[["has_valid_gpa", "has_valid_gre", "has_valid_gre_v",
         "days_to_decision", "decision_speed", "application_season"]].head()

,has_valid_gpa,has_valid_gre,has_valid_gre_v,days_to_decision,decision_speed,application_season
0,1,0,0,-112,Unknown,Late Cycle
1,1,0,0,0,0-30 days,Late Cycle
2,0,0,0,-13,Unknown,Late Cycle
3,0,0,0,-3,Unknown,Late Cycle
4,1,0,0,-14,Unknown,Late Cycle


## 5. Summary statistics & outlier analysis

Descriptive statistics for each numeric column, saved as
`summary_statistics.csv`.

In [15]:
def describe_numeric(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    return {
        "count": int(s.count()), "mean": s.mean(), "median": s.median(),
        "std": s.std(), "min": s.min(), "max": s.max(),
        "25%": s.quantile(0.25), "50%": s.quantile(0.50), "75%": s.quantile(0.75),
    }

summary_stats = pd.DataFrame(
    {c: describe_numeric(cleaned[c]) for c in ["GPA", "GRE", "GRE V", "GRE AW"]}
).T.round(3)
summary_stats.to_csv("summary_statistics.csv")
summary_stats

,count,mean,median,std,min,max,25%,50%,75%
GPA,17557.0,3.791,3.85,0.365,0.1,9.99,3.68,3.85,3.96
GRE,2395.0,260.401,316.00,86.849,4.0,999.00,168.00,316.00,328.00
GRE V,2004.0,161.583,161.00,27.610,4.0,999.00,157.00,161.00,166.00
GRE AW,1882.0,8.423,4.50,19.314,2.0,99.99,4.00,4.50,5.00


### Outliers via `scipy.stats.zscore`

Flag values whose **|z-score| > 3** in GPA and GRE (`nan_policy="omit"` so
missing values are ignored). We **keep** the flagged rows — they are recorded in
Boolean columns rather than deleted, so downstream analysis can filter on them
without losing data. Saved as `outlier_summary.png`.

In [16]:
def zscore_outliers(series, thresh=3.0):
    z = pd.Series(np.nan, index=series.index)
    mask = series.notna()
    z[mask] = stats.zscore(series[mask].to_numpy(), nan_policy="omit")
    return (z.abs() > thresh).fillna(False)

cleaned["gpa_outlier"] = zscore_outliers(cleaned["GPA"])
cleaned["gre_outlier"] = zscore_outliers(cleaned["GRE"])

outlier_summary = pd.DataFrame({
    "column": ["GPA", "GRE"],
    "n_values": [int(cleaned["GPA"].notna().sum()), int(cleaned["GRE"].notna().sum())],
    "n_outliers": [int(cleaned["gpa_outlier"].sum()), int(cleaned["gre_outlier"].sum())],
    "rule": ["|z| > 3", "|z| > 3"],
    "decision": ["kept (flagged, not dropped)", "kept (flagged, not dropped)"],
})
gradcafe.render_table_png(outlier_summary, "outlier_summary.png",
                          title="Outlier summary (z-score > 3)",
                          max_rows=len(outlier_summary), index=False)
outlier_summary

,column,n_values,n_outliers,rule,decision
0,GPA,17557,83,|z| > 3,"kept (flagged, not dropped)"
1,GRE,2395,4,|z| > 3,"kept (flagged, not dropped)"


## 6. Statistical analysis with SciPy

**A. Correlation** — Pearson on non-zero GRE vs GRE V, and Pearson on valid GPA
(0-4) vs non-zero GRE.

In [17]:
gg = cleaned[(cleaned["GRE"] > 0) & (cleaned["GRE V"] > 0)][["GRE", "GRE V"]].dropna()
gre_grev_r, gre_grev_p = stats.pearsonr(gg["GRE"], gg["GRE V"])

pg = cleaned[(cleaned["GPA"].between(0, 4.0)) & (cleaned["GRE"] > 0)][["GPA", "GRE"]].dropna()
gpa_gre_r, gpa_gre_p = stats.pearsonr(pg["GPA"], pg["GRE"])

print(f"GRE vs GRE V : Pearson r = {gre_grev_r:.3f}, p = {gre_grev_p:.3g}, n = {len(gg)}")
print(f"GPA vs GRE   : Pearson r = {gpa_gre_r:.3f}, p = {gpa_gre_p:.3g}, n = {len(pg)}")

GRE vs GRE V : Pearson r = 0.284, p = 1.06e-36, n = 1900
GPA vs GRE   : Pearson r = -0.001, p = 0.954, n = 2098


**B. Group comparison** — GPA of Accepted vs Rejected applicants (valid GPAs
only), using Welch's two-sample t-test (`ttest_ind`, `equal_var=False`).

In [18]:
acc = cleaned[(cleaned["outcome"] == "Accepted") &
              (cleaned["GPA"].between(0, 4.0))]["GPA"].dropna()
rej = cleaned[(cleaned["outcome"] == "Rejected") &
              (cleaned["GPA"].between(0, 4.0))]["GPA"].dropna()
group_stat, group_p = stats.ttest_ind(acc, rej, equal_var=False)
print(f"Welch t-test: t = {group_stat:.3f}, p = {group_p:.3g}")
print(f"Accepted mean GPA = {acc.mean():.3f} (n={len(acc)}) | "
      f"Rejected mean GPA = {rej.mean():.3f} (n={len(rej)})")

Welch t-test: t = -4.831, p = 1.37e-06
Accepted mean GPA = 3.763 (n=7102) | Rejected mean GPA = 3.782 (n=7986)


**C. Categorical association** — chi-square test of **Degree vs
US/International** (`chi2_contingency`).

In [19]:
contingency = pd.crosstab(cleaned["Degree"], cleaned["US/International"])
chi2_stat, chi2_p, chi2_dof, chi2_expected = stats.chi2_contingency(contingency)
print("Contingency table:")
print(contingency)
print(f"\nchi2 = {chi2_stat:.3f}, p = {chi2_p:.3g}, dof = {chi2_dof}")

Contingency table:
US/International  American  International  Other
Degree                                          
Master's              4500           2751    310
PhD                  10166          10794    316
PsyD                   277             20      7

chi2 = 772.681, p = 6.35e-166, dof = 4


## 7. Visualizations

Six labelled Matplotlib figures, each saved as its own PNG. The two scatter plots
include a NumPy `polyfit` best-fit line in the legend.

In [20]:
gradcafe.scatter_gre_vs_grev(cleaned, "GRE-vs-GRE-V.png")
gradcafe.scatter_gpa_vs_gre(cleaned, "GPA-vs-GRE.png")
gradcafe.bar_degree_by_international(cleaned, "Degree-vs-International.png")
gradcafe.hist_acceptances_over_time(cleaned, "Acceptances-over-Time.png")
gradcafe.box_gpa_by_outcome(cleaned, "GPA-by-Outcome.png")
gradcafe.correlation_heatmap(cleaned, "Numeric-Correlation-Heatmap.png")
print("Saved 6 figures: GRE-vs-GRE-V, GPA-vs-GRE, Degree-vs-International,")
print("                 Acceptances-over-Time, GPA-by-Outcome, Numeric-Correlation-Heatmap")

Saved 6 figures: GRE-vs-GRE-V, GPA-vs-GRE, Degree-vs-International,
                 Acceptances-over-Time, GPA-by-Outcome, Numeric-Correlation-Heatmap


## 8. Analytical summary PDF

Compute the answer values from the cleaned data, then render `analytics.pdf`.
Interpretation text is derived from the computed statistics (significance keyed to
the p-values) — nothing is hard-coded.

In [21]:
avg_gre_accepted_phd = cleaned[(cleaned["outcome"] == "Accepted") &
    (cleaned["Degree"] == "PhD") & (cleaned["GRE"] > 0)]["GRE"]
avg_gpa_accepted_masters = cleaned[(cleaned["outcome"] == "Accepted") &
    (cleaned["Degree"] == "Master's") & (cleaned["GPA"].between(0, 4.0))]["GPA"]

def sig(p):
    return "statistically significant" if p < 0.05 else "not statistically significant"

corr_interpretation = (
    f"GRE and GRE V are {'positively' if gre_grev_r > 0 else 'negatively'} correlated "
    f"(r={gre_grev_r:.2f}), a relationship that is {sig(gre_grev_p)}. GPA and GRE show a "
    f"{'weak' if abs(gpa_gre_r) < 0.3 else 'moderate-to-strong'} "
    f"{'positive' if gpa_gre_r > 0 else 'negative'} association ({sig(gpa_gre_p)}).")
group_interpretation = (
    f"Accepted applicants average {acc.mean():.2f} GPA vs {rej.mean():.2f} for rejected; "
    f"the difference is {sig(group_p)} (p={group_p:.3g}), so accepted and rejected "
    f"applicants {'do' if group_p < 0.05 else 'do not clearly'} differ in reported GPA.")
chi2_interpretation = (
    f"The association between Degree and US/International is {sig(chi2_p)} "
    f"(p={chi2_p:.3g}); Degree type and applicant origin are "
    f"{'not independent' if chi2_p < 0.05 else 'plausibly independent'}.")

stats_for_pdf = {
    "raw_rows": n_raw, "clean_rows": len(cleaned), "clean_cols": cleaned.shape[1],
    "avg_gre_accepted_phd": round(float(avg_gre_accepted_phd.mean()), 2),
    "n_gre_accepted_phd": int(avg_gre_accepted_phd.count()),
    "avg_gpa_accepted_masters": round(float(avg_gpa_accepted_masters.mean()), 3),
    "n_gpa_accepted_masters": int(avg_gpa_accepted_masters.count()),
    "gre_grev_r": round(float(gre_grev_r), 3), "gre_grev_p": f"{gre_grev_p:.3g}",
    "gre_grev_n": len(gg),
    "gpa_gre_method": "Pearson", "gpa_gre_r": round(float(gpa_gre_r), 3),
    "gpa_gre_p": f"{gpa_gre_p:.3g}", "gpa_gre_n": len(pg),
    "corr_interpretation": corr_interpretation,
    "group_test": "Welch two-sample t-test (ttest_ind, equal_var=False)",
    "group_stat": round(float(group_stat), 3), "group_p": f"{group_p:.3g}",
    "accepted_gpa_mean": round(float(acc.mean()), 3), "accepted_gpa_n": len(acc),
    "rejected_gpa_mean": round(float(rej.mean()), 3), "rejected_gpa_n": len(rej),
    "group_interpretation": group_interpretation,
    "chi2_stat": round(float(chi2_stat), 3), "chi2_p": f"{chi2_p:.3g}",
    "chi2_dof": int(chi2_dof), "chi2_interpretation": chi2_interpretation,
}
gradcafe.build_analytics_pdf("analytics.pdf", stats_for_pdf)
print("Wrote analytics.pdf")
print("Q1 avg GRE of accepted PhD applicants :",
      stats_for_pdf["avg_gre_accepted_phd"])
print("Q2 avg GPA of accepted Master's applicants:",
      stats_for_pdf["avg_gpa_accepted_masters"])

Wrote analytics.pdf
Q1 avg GRE of accepted PhD applicants : 261.81
Q2 avg GPA of accepted Master's applicants: 3.719


## 9. Export the cleaned dataset and return it to S3

Write `cleaned_gradcafe.json`, then upload it back to the bucket with the
Module 7 `boto3` workflow (`s3_fetch.upload_dataset`).

In [22]:
clean_out = "cleaned_gradcafe.json"
cleaned.to_json(clean_out, orient="records", date_format="iso", indent=2)
print(f"Wrote {clean_out}: {len(cleaned)} records, {cleaned.shape[1]} columns")

s3_uri = s3_fetch.upload_dataset(clean_out)
print(f"Upload succeeded -> {s3_uri}")

Wrote cleaned_gradcafe.json: 29141 records, 37 columns
Upload succeeded -> s3://grad-cafe-rg/cleaned_gradcafe.json


## 10. Shut down reminder

Analysis complete and the cleaned dataset is back in S3. **Remember to STOP this
SageMaker notebook instance** (Notebook instances → Stop) so it stops incurring
charges. Stopping (not terminating) preserves the instance for reuse.